In [ ]:
import pandas as pd
import numpy as np


df = pd.DataFrame({
    "record_id": [
        101, 102, 103, 104, 105,
        106, 107, 108, 109, 110
    ],

    "site": [
        " R34 ", "R35", " R36", "R34 ",
        " R35 ", "R36", "R34", " R35",
        "R36 ", " R34"
    ],

    "model": [
        "fs11", "FS11", "Fs11", "FS11",
        "fs11", "FS11", "Fs11", "fs11",
        "FS11", "Fs11"
    ],

    "status": [
        "OK", "ERROR", "NORMAL", "OK",
        "ERROR", "NORMAL", "OK", "ERROR",
        "NORMAL", "OK"
    ],

    "sensor_data": [
        "VIS 73621 AL 0",
        "VIS 45800 AL 0",
        "ERROR",
        "VIS 61200 AL 0",
        "VIS 38950 AL 0",
        "NO DATA",
        "VIS 80120 AL 0",
        "VIS 52300 AL 0",
        "VIS 67500 AL 0",
        np.nan
    ],

    "remark": [
        "normal communication",
        "connection timeout",
        np.nan,
        "Normal",
        "sensor disconnect",
        "NO RESPONSE",
        "normal",
        "Timeout detected",
        "communication normal",
        "DISCONNECT"
    ]
})

df

## 任务 1：清洗站点编号

`site` 字段存在前后空格。

清洗该字段，使：`" R34 "`、`"R34"`、`" R34"`

最终统一为：`"R34"`

不要创建新的站点字段，直接更新 `site`。


In [ ]:
df_cleaned = df.copy()

In [ ]:
df_cleaned['site'] = df_cleaned['site'].str.strip()
df_cleaned['site']

## 任务 2：统一设备型号

`model` 字段存在大小写不统一的问题，例如：`"fs11"`、`"FS11"`、`"Fs11"`

将所有型号统一为大写形式。

In [ ]:
df_cleaned['model'] = df_cleaned['model'].str.upper()
df_cleaned['model']

## 任务 3：统一状态字段

`status` 中：

`"OK"`、`"NORMAL"` 业务含义相同。

统一将 `"OK"` 修改为 `"NORMAL"`。

其它状态保持不变。

In [ ]:
df_cleaned['status'] = df_cleaned['status'].replace('OK','NORMAL')
df_cleaned['status']

## 任务 4：提取设备原始数据中的 VIS

`sensor_data` 中包含类似：

`"VIS 73621 AL 0"`

这样的原始字符串。

从中提取 `VIS` 后面的数字，并生成新列：`sensor_vis`

要求最终 `sensor_vis` 是数值类型，而不是字符串。

无法提取 VIS 的记录允许为缺失值。

In [ ]:
df_cleaned['sensor_vis'] =(
    df_cleaned['sensor_data']
    .str.extract(r'VIS\s+(\d+)',expand=False)
    .pipe(pd.to_numeric,errors='coerce')
    .astype('Int64')
)
df_cleaned['sensor_vis']

## 任务 5：识别可能的通信故障

检查 `remark` 字段。

只要备注中包含：`timeout` 或者：`disconnect` 就认为存在通信异常。

要求：

- 不区分大小写；
- `remark` 缺失时不能报错。

生成布尔列：`communication_error`



In [ ]:
df_cleaned['communication_error'] = (
    df_cleaned['remark']
    .str
    .contains(r'timeout|disconnect',case=False,na=False)
)
df_cleaned['communication_error']

## 任务 6：生成最终分析数据

最终只保留：

- record_id
- site
- model
- status
- sensor_vis
- communication_error

保存为：

```python
result
```

In [ ]:
result = df_cleaned[['record_id','site','model','status','sensor_vis','communication_error']]
result